In [1]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_chroma import Chroma
from langchain_classic.retrievers import EnsembleRetriever
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_community.document_compressors.flashrank_rerank import FlashrankRerank
from langchain_community.document_loaders import ArxivLoader
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_mistralai import ChatMistralAI, MistralAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pydantic import BaseModel, Field

load_dotenv()

True

In [2]:
vector_store= Chroma(
    embedding_function=MistralAIEmbeddings(),
    persist_directory="Research_Papers",
    collection_name="Arxiv_papers",
)

In [3]:
class load_and_split_input(BaseModel):
    paper_identifier: str= Field(description="Unique numerical identifier for the research paper.")
    chunk_size: int= Field(description="Size of each chunk, dynamically decide it based on the length of the paper.", gt= 500)

@tool
def load_split_and_store(paper_identifier: str, chunk_size: int) -> str:
    """ Use this tool to load the research paper when a unique numerical 
    identifier is provided. It checks if the paper is already in the database. 
    """

    existing_docs = vector_store.get(where={"paper_id": paper_identifier}, limit=1)
    
    if existing_docs['ids']:
        return f"Paper {paper_identifier} is already in the Vector store. Proceed to search or summarizer."

    loader = ArxivLoader(query=paper_identifier, load_max_docs=1)
    docs = loader.load()
    
    if not docs:
        return "Could not find a paper with that identifier on Arxiv."

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=100,
    )
    chunks = splitter.split_documents(docs)

    for chunk in chunks:
        chunk.metadata["paper_id"] = paper_identifier

    vector_store.add_documents(chunks)
    
    return f"Paper {paper_identifier} loaded and indexed successfully."

In [4]:
try:
    from flashrank import Ranker, RerankRequest
    FlashrankRerank.model_rebuild(_types_namespace={
        'Ranker': Ranker, 
        'RerankRequest': RerankRequest
    })
except ImportError:
    print("Please install flashrank: pip install flashrank")

compressor = FlashrankRerank()

def hybrid_retrieve(query: str, paper_identifier: str) -> str:

    docs = vector_store.get(where={"paper_id": str(paper_identifier)})
    
    if not docs or not docs['documents']:
        return "No content found for this paper ID."

    paper_chunks = [
        Document(page_content=text, metadata=meta) 
        for text, meta in zip(docs['documents'], docs['metadatas'])
    ]

    bm25_retriever = BM25Retriever.from_documents(paper_chunks)
    bm25_retriever.k = 5
    
    vector_retriever = vector_store.as_retriever(
        search_kwargs={"k": 5, "filter": {"paper_id": str(paper_identifier)}}
    )

    ensemble = EnsembleRetriever(
        retrievers=[bm25_retriever, vector_retriever], 
        weights=[0.3, 0.7]
    )

    compression_retriever = ContextualCompressionRetriever(
        base_compressor=compressor, 
        base_retriever=ensemble
    )

    compressed_docs = compression_retriever.invoke(query)

    if not compressed_docs:
        return "Search completed, but no relevant information was found."

    return "\n\n".join([f"Content: {d.page_content}" for d in compressed_docs])

@tool
def search_paper_content(query: str, paper_identifier: str) -> str:
    """
    Useful for searching specific details within a research paper.
    Query should be the search string, paper_identifier is the ID (e.g., '1706.03762').
    """

    return hybrid_retrieve(query, str(paper_identifier))

In [5]:
@tool
def summarizer(arxiv_id: str) -> str:
    """ Use this tool to fetch the entire research paper *only when explicitly requested* to summarize it. """
    loader= ArxivLoader(query= arxiv_id, load_max_docs= 3, load_all_available_meta=False)
    docs= loader.load()
    full_text= " ".join([doc.page_content for doc in docs])
    return full_text

In [6]:
model= ChatMistralAI(model="mistral-medium-latest")

In [7]:
agent= create_agent(
    model= model,
    tools= [load_split_and_store, search_paper_content, summarizer],
    system_prompt="""
    You are a highly skilled researcher with knowledge of all research papers in the field of computer science domain. 
    You have access to tools called load_split_and_store, search_paper and summarizer. 
    You can use these tools to load the research paper when a unique numerical identifier is provided.
    You can use these tools to fetch the entire research paper or summarize it when explicitly requested.
    """
)

In [10]:
paper_identifier=2503.07137

# Entire summary

In [8]:
query= f"Summarize the entire research paper with paper_identifier: {paper_identifier}"
result = agent.invoke({
    "messages": [
        {"role": "user", "content": query}
    ]
})
result

INFO:httpx:HTTP Request: POST https://api.mistral.ai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:arxiv:Requesting page (first: True, try: 0): https://export.arxiv.org/api/query?search_query=&id_list=2503.07137&sortBy=relevance&sortOrder=descending&start=0&max_results=100
INFO:arxiv:Got first page: 1 of 1 total results
INFO:httpx:HTTP Request: POST https://api.mistral.ai/v1/chat/completions "HTTP/1.1 200 OK"


{'messages': [HumanMessage(content='Summarize the entire research paper with paper_identifier: 2503.07137', additional_kwargs={}, response_metadata={}, id='4447ad0c-0a98-4e28-b110-5360f3d7efce'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'lnUOmvfxW', 'function': {'name': 'summarizer', 'arguments': '{"arxiv_id": "2503.07137"}'}, 'index': 0}]}, response_metadata={'token_usage': {'prompt_tokens': 398, 'total_tokens': 422, 'completion_tokens': 24, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-medium-latest', 'model': 'mistral-medium-latest', 'finish_reason': 'tool_calls', 'model_provider': 'mistralai'}, id='lc_run--019ce06f-3dc0-77c2-92c5-b0d557b54f1d-0', tool_calls=[{'name': 'summarizer', 'args': {'arxiv_id': '2503.07137'}, 'id': 'lnUOmvfxW', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 398, 'output_tokens': 24, 'total_tokens': 422}),
  ToolMessage(content='1\nA Comprehensive Survey of Mixture-of-Experts:\nAl

In [9]:
print(result['messages'][-1].content)

The research paper titled **"A Comprehensive Survey of Mixture-of-Experts: Algorithms, Theory, and Applications"** by Siyuan Mu and Sen Lin provides an in-depth exploration of the **Mixture-of-Experts (MoE)** paradigm in artificial intelligence. Below is a structured summary of the paper:

---

### **Abstract**
The paper addresses the challenges faced by large AI models, particularly in terms of **computational resource consumption** and **handling diverse and complex data**. MoE models offer a solution by dynamically selecting and activating the most relevant sub-models (experts) for processing input data. This approach improves **model performance** and **efficiency** while reducing computational costs. The survey covers:
- Basic design principles of MoE.
- Algorithm design in machine learning paradigms like **continual learning, meta-learning, multi-task learning, reinforcement learning, and federated learning**.
- Theoretical studies on MoE.
- Applications in **computer vision (CV)

# loading and retrieving steps

In [15]:
query= f"What does the paper suggest for NLP tasks? paper_identifier: {paper_identifier}"
result = agent.invoke({
    "messages": [
        {"role": "user", "content": query}
    ]
})
result

INFO:httpx:HTTP Request: POST https://api.mistral.ai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.mistral.ai/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.mistral.ai/v1/chat/completions "HTTP/1.1 200 OK"


{'messages': [HumanMessage(content='What does the paper suggest for NLP tasks? paper_identifier: 2503.07137', additional_kwargs={}, response_metadata={}, id='379878b4-e023-4027-9f75-feb152920312'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'PHHjl38sW', 'function': {'name': 'search_paper_content', 'arguments': '{"query": "NLP tasks", "paper_identifier": "2503.07137"}'}, 'index': 0}]}, response_metadata={'token_usage': {'prompt_tokens': 400, 'total_tokens': 432, 'completion_tokens': 32, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-medium-latest', 'model': 'mistral-medium-latest', 'finish_reason': 'tool_calls', 'model_provider': 'mistralai'}, id='lc_run--019ce079-4c30-7363-9ab7-1e13c5cb64ff-0', tool_calls=[{'name': 'search_paper_content', 'args': {'query': 'NLP tasks', 'paper_identifier': '2503.07137'}, 'id': 'PHHjl38sW', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 400, 'output_tokens': 32, 'total_tokens': 

In [16]:
print(result['messages'][-1].content)

The paper **2503.07137** discusses the application of **Mixture of Experts (MoE)** architectures in **NLP tasks**, emphasizing their potential to enhance model performance, specialization, and efficiency. Here are the key suggestions and insights for NLP tasks:

---

### **1. Natural Language Generation (NLG)**
- **Text Generation Challenges**: Traditional NLP systems face issues like **insufficient generation diversity** and **uncontrollable generated content**.
- **MoE Solutions**:
  - MoE dynamically selects the most suitable **sub-network (expert)** for specific tasks, improving efficiency and precision in processing complex or diverse text data.
  - In **language GANs**, MoE is used in the generator to collaboratively generate high-quality sentences. Each expert acts as a **recurrent neural network**, autoregressively producing token representations.
  - This approach enhances **text generation quality** and **diversity**.

---

### **2. Natural Language Understanding (NLU)**
- **